# vgc-ai: PPO self-play on a Kaggle GPU

Trains the policy/value network of [vgc-ai](https://github.com/zeronohako/vgc-ai) (`train.py`) on a Kaggle GPU.
The battles run on the CPU; the GPU trains the network.

**Settings (right panel):** Accelerator **GPU T4 x2** (one T4 is used; not the TPU, not the P100). Internet **on**
(needs a phone-verified account).

**Run it in the background:** *Save Version* → *Save & Run All (Commit)*. The browser can be closed. The run folder
(checkpoint, log, snapshots) is saved in the notebook's output under `runs/<RUN>/`.

**Next session:** *Add Input* → *Your Work* → this notebook (its latest output), then *Save & Run All* again. The
setup cell finds the newest checkpoint of `RUN` under `/kaggle/input` and continues from it; if `RUN` has none
yet, it starts from `FROM_RUN`'s (its checkpoint, log and snapshots are copied into `runs/<RUN>/`).

**Back home:** download `runs/<RUN>/` from the output into the repo's `runs/` and measure it with `arena.py`.

In [ ]:
RUN = "kaggle-d128-pfsp"     # run name: the output folder runs/<RUN>/
FROM_RUN = "kaggle-d128"     # if RUN has no checkpoint among the inputs yet, start from this run's ("": a new run)
MINUTES = 420                # training minutes this session; sessions are capped at ~9-12 h, so keep a margin
NET = "--d 128 --layers 3"   # network size (a new run only; a resumed run must keep its size)
EXTRA = "--eval-every 25 --snapshot-every 50 --pfsp 0.5 --target-kl 0.01"  # half the battles vs own snapshots; lr by KL
BENCHMARK = False            # True: first 3 min with moves picked on the GPU, 3 min on the CPU; train with the faster
REPO, BRANCH = "https://github.com/zeronohako/vgc-ai.git", "main"

In [ ]:
# Environment: code, Python 3.12+ (ninjask's wheel), ninjask; Kaggle's own CUDA build of torch is kept.
import glob, json, os, shutil, subprocess, sys
print(sys.version.split()[0], "|", os.cpu_count(), "CPUs")
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"])
shutil.rmtree("/tmp/vgc-ai", ignore_errors=True)  # a re-run clones afresh (git won't clone into an existing folder)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "/tmp/vgc-ai"], check=True)
subprocess.run(["git", "-C", "/tmp/vgc-ai", "log", "--oneline", "-1"])
assert os.path.exists("/tmp/vgc-ai/train.py"), "GitHub's main has no train.py yet: push the local commits first"
if sys.version_info >= (3, 12):
    PY = sys.executable
    subprocess.run([PY, "-m", "pip", "install", "-q", "ninjask==0.4.0"], check=True)
else:  # older image: a Python 3.12 venv, torch matched to the GPU driver by uv
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "venv", "-q", "-p", "3.12", "/tmp/venv"], check=True)
    PY = "/tmp/venv/bin/python"
    subprocess.run(["uv", "pip", "install", "-q", "-p", PY, "--torch-backend=auto", "torch", "numpy", "ninjask==0.4.0"],
                   check=True)
subprocess.run([PY, "-c", "import torch, ninjask; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),"
                " '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')"], check=True)

# Resume: the newest checkpoint of RUN among the inputs (a previous session's output), else FROM_RUN's.
def newest(run):
    best, best_it = None, -1
    for ck in glob.glob(f"/kaggle/input/**/runs/{run}/ckpt.pt", recursive=True):
        folder = os.path.dirname(ck)
        lines = open(os.path.join(folder, "log.jsonl")).read().splitlines()
        it = json.loads(lines[-1])["it"] if lines else 0
        if it > best_it:
            best, best_it = folder, it
    return best, best_it

best, best_it = newest(RUN)
if best is None and FROM_RUN:
    best, best_it = newest(FROM_RUN)
    assert best, f"no checkpoint of {RUN} or {FROM_RUN} under /kaggle/input: Add Input -> Your Work -> its output"
RESUME = f"--resume-from {best}" if best else ""
print(f"resume from {best} at iteration {best_it}" if best else f"new run {RUN}")

In [ ]:
# Smoke test on the GPU (~1 min): the run starts, trains, evaluates and (--pfsp in EXTRA) plays its own snapshots.
!cd /tmp/vgc-ai && {PY} train.py --run smoke --runs-dir /tmp/smoke --minutes 0.5 --battles 64 {NET} {EXTRA} --eval-every 2 --eval-games 32 --snapshot-every 2

In [ ]:
# Benchmark (~7 min): decisions per second with moves picked on the GPU vs on a CPU copy of the network.
ACT = "cuda"
if BENCHMARK:
    rates = {}
    for act in ("cuda", "cpu"):
        out = f"/tmp/bench-{act}"
        subprocess.run(f"cd /tmp/vgc-ai && {PY} train.py --run b --runs-dir {out} --minutes 3 --eval-every 1000 "
                       f"--act-device {act} {NET}", shell=True, check=True, capture_output=True)
        recs = [json.loads(l) for l in open(f"{out}/b/log.jsonl")][2:]  # skip the untrained eval and a warm-up
        rates[act] = sum(r["per_s"] for r in recs) / max(len(recs), 1)
        print(f"moves on {act}: {rates[act]:.0f} decisions/s over {len(recs)} iterations "
              f"(collect {sum(r['collect_s'] for r in recs) / max(len(recs), 1):.1f} s, "
              f"train {sum(r['train_s'] for r in recs) / max(len(recs), 1):.1f} s per iteration)")
    ACT = max(rates, key=rates.get)
print("moves on", ACT)

In [ ]:
# Train. Output: /kaggle/working/runs/<RUN>/ (ckpt.pt, log.jsonl, snap_*.pt).
!cd /tmp/vgc-ai && {PY} train.py --run {RUN} --runs-dir /kaggle/working/runs {RESUME} --minutes {MINUTES} --act-device {ACT} {NET} {EXTRA}

In [ ]:
# Summary of this run so far.
recs = [json.loads(l) for l in open(f"/kaggle/working/runs/{RUN}/log.jsonl")]
last = recs[-1]
print(f"iteration {last['it']}, {last['minutes']:.0f} min trained, {last['decisions']:,} decisions, {last['games']:,} games")
evals = [(r["it"], r["vs_random"]) for r in recs if "vs_random" in r]
print("vs random (last 5 evals):", evals[-5:])
lg = [r["league"] for r in recs[-25:] if r.get("league") is not None]
if lg:
    hardest = next((r["hardest"] for r in reversed(recs) if "hardest" in r), None)
    print(f"vs its own snapshots (last {len(lg)} iterations): {sum(lg) / len(lg):.3f}; hardest [it, score]: {hardest}")
print(f"lr {last.get('lr')}, KL {last['kl']}, collect {last['collect_s']} s + train {last['train_s']} s per iteration")
print(len(glob.glob(f"/kaggle/working/runs/{RUN}/snap_*.pt")), "snapshots")